In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [2]:
df = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


<h2> 1. Lowercasing </h2>

In [4]:
df['review'][3].lower()

"basically there's a family where a little boy (jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />this movie is slower than a soap opera... and suddenly, jake decides to become rambo and kill the zombie.<br /><br />ok, first of all when you're going to make a film you must decide if its a thriller or a drama! as a drama the movie is watchable. parents are divorcing & arguing like in real life. and then we have jake with his closet which totally ruins all the film! i expected to see a boogeyman similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. as for the shots with jake: just ignore them."

In [5]:
df['review'] = df['review'].str.lower()

In [6]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative


<h2> 2. Remove HTML Tags </h2>

In [7]:
text = "<p>Welcome to <strong>NLP text cleaning</strong>.</p><br>Visit the <a href='https://example.com'>example website</a> for more info."

In [8]:
import re 

def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

In [9]:
remove_html_tags(text)

'Welcome to NLP text cleaning.Visit the example website for more info.'

In [10]:
df['review'] = df['review'].apply(remove_html_tags)

In [11]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative


In [12]:
df['review'][3]

"basically there's a family where a little boy (jake) thinks there's a zombie in his closet & his parents are fighting all the time.this movie is slower than a soap opera... and suddenly, jake decides to become rambo and kill the zombie.ok, first of all when you're going to make a film you must decide if its a thriller or a drama! as a drama the movie is watchable. parents are divorcing & arguing like in real life. and then we have jake with his closet which totally ruins all the film! i expected to see a boogeyman similar movie, and instead i watched a drama with some meaningless thriller spots.3 out of 10 just for the well playing parents & descent dialogs. as for the shots with jake: just ignore them."

<h2> 3. Remove URL tags</h2>

In [13]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)

In [14]:
text1 = "Check out my notebook https://www.kaggle.com/campusx/notebook8223fc1abb"	
text2 = "Google search here www.google.com"	
text3 = "For notebook click https://www.kaggle.com/campusx/notebook8223fc1abb to search check www.google.com"

In [15]:
print(remove_url(text1))
print(remove_url(text2))
print(remove_url(text3))

Check out my notebook 
Google search here 
For notebook click  to search check 


<h2> 4. Remove Punctuation </h2>

In [16]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [17]:
exclude = string.punctuation

In [18]:
def remove_punc(text):
    for char in exclude:
        text = text.replace(char, '')
    return text    

In [19]:
text = "Hello, World! Python is amazing."

In [20]:
import time 

start = time.time()
print(remove_punc(text))

time1 = time.time() - start
print(time1*50000)

Hello World Python is amazing
29.69503402709961


In [21]:
# Advance way for "removal_of_punctuation" :- 

def remove_punc1(text):
    return text.translate(str.maketrans('','',exclude))

In [22]:
start = time.time()

print(remove_punc1(text));
time2 = time.time() - start;
print(time2*50000)

Hello World Python is amazing
14.293193817138672


In [23]:
print(time1/time2)

2.0775646371976646


<h2> 5. Chat word treatment </h2>

In [24]:
chat_words = {
    "A3": "Anytime, Anywhere, Anyplace", "ADIH": "Another Day In Hell", 
    "AFK": "Away From Keyboard", "AFAIK": "As Far As I Know", 
    "ASAP": "As Soon As Possible", "ASL": "Age, Sex, Location", 
    "ATK": "At The Keyboard", "ATM": "At The Moment", 
    "BAE": "Before Anyone Else", "BAK": "Back At Keyboard", 
    "BBL": "Be Back Later", "BBS": "Be Back Soon", 
    "BFN": "Bye For Now", "B4N": "Bye For Now", 
    "BRB": "Be Right Back", "BRUH": "Bro", 
    "BRT": "Be Right There", "BSAAW": "Big Smile And A Wink", 
    "BTW": "By The Way", "BWL": "Bursting With Laughter", 
    "CSL": "Can’t Stop Laughing", "CU": "See You", 
    "CUL8R": "See You Later", "CYA": "See You", 
    "DM": "Direct Message", "FAQ": "Frequently Asked Questions", 
    "FC": "Fingers Crossed", "FIMH": "Forever In My Heart", 
    "FOMO": "Fear Of Missing Out", "FR": "For Real", 
    "FWIW": "For What It's Worth", "FYP": "For You Page", 
    "FYI": "For Your Information", "G9": "Genius", 
    "GAL": "Get A Life", "GG": "Good Game", 
    "GMTA": "Great Minds Think Alike", "GN": "Good Night", 
    "GOAT": "Greatest Of All Time", "GR8": "Great!", 
    "HBD": "Happy Birthday", "IC": "I See", 
    "ICQ": "I Seek You", "IDC": "I Don’t Care", 
    "IDK": "I Don't Know", "IFYP": "I Feel Your Pain", 
    "ILU": "I Love You", "ILY": "I Love You", 
    "IMHO": "In My Honest/Humble Opinion", "IMU": "I Miss You", 
    "IMO": "In My Opinion", "IOW": "In Other Words", 
    "IRL": "In Real Life", "IYKYK": "If You Know, You Know", 
    "JK": "Just Kidding", "KISS": "Keep It Simple, Stupid", 
    "L": "Loss", "L8R": "Later", "LDR": "Long Distance Relationship", 
    "LMK": "Let Me Know", "LMAO": "Laughing My A** Off", 
    "LOL": "Laughing Out Loud", "LTNS": "Long Time No See", 
    "M8": "Mate", "MFW": "My Face When", "MID": "Mediocre", 
    "MRW": "My Reaction When", "MTE": "My Thoughts Exactly", 
    "NVM": "Never Mind", "NRN": "No Reply Necessary", 
    "NPC": "Non-Player Character", "OIC": "Oh I See", 
    "OP": "Overpowered", "PITA": "Pain In The A**", 
    "POV": "Point Of View", "PRT": "Party", "PRW": "Parents Are Watching", 
    "ROFL": "Rolling On The Floor Laughing", 
    "ROFLOL": "Rolling On The Floor Laughing Out Loud", 
    "ROTFLMAO": "Rolling On The Floor Laughing My A** Off", 
    "RN": "Right Now", "SK8": "Skate", "STATS": "Your Sex And Age", 
    "SUS": "Suspicious", "TBH": "To Be Honest", 
    "TFW": "That Feeling When", "THX": "Thank You", 
    "TIME": "Tears In My Eyes", "TLDR": "Too Long, Didn’t Read", 
    "TNTL": "Trying Not To Laugh", "TTFN": "Ta-Ta For Now!", 
    "TTYL": "Talk To You Later", "U": "You", "U2": "You Too", 
    "U4E": "Yours For Ever", "W": "Win", "W8": "Wait...", 
    "WB": "Welcome Back", "WTF": "What The F**k", 
    "WTG": "Way To Go!", "WUF": "Where Are You From?", 
    "WYD": "What You Doing?", "WYWH": "Wish You Were Here", 
    "ZZZ": "Sleeping, Bored, Tired"
}

In [25]:
chat_words

{'A3': 'Anytime, Anywhere, Anyplace',
 'ADIH': 'Another Day In Hell',
 'AFK': 'Away From Keyboard',
 'AFAIK': 'As Far As I Know',
 'ASAP': 'As Soon As Possible',
 'ASL': 'Age, Sex, Location',
 'ATK': 'At The Keyboard',
 'ATM': 'At The Moment',
 'BAE': 'Before Anyone Else',
 'BAK': 'Back At Keyboard',
 'BBL': 'Be Back Later',
 'BBS': 'Be Back Soon',
 'BFN': 'Bye For Now',
 'B4N': 'Bye For Now',
 'BRB': 'Be Right Back',
 'BRUH': 'Bro',
 'BRT': 'Be Right There',
 'BSAAW': 'Big Smile And A Wink',
 'BTW': 'By The Way',
 'BWL': 'Bursting With Laughter',
 'CSL': 'Can’t Stop Laughing',
 'CU': 'See You',
 'CUL8R': 'See You Later',
 'CYA': 'See You',
 'DM': 'Direct Message',
 'FAQ': 'Frequently Asked Questions',
 'FC': 'Fingers Crossed',
 'FIMH': 'Forever In My Heart',
 'FOMO': 'Fear Of Missing Out',
 'FR': 'For Real',
 'FWIW': "For What It's Worth",
 'FYP': 'For You Page',
 'FYI': 'For Your Information',
 'G9': 'Genius',
 'GAL': 'Get A Life',
 'GG': 'Good Game',
 'GMTA': 'Great Minds Think Alik

In [26]:
def chat_conversion(text):
    new_text = []

    for w in text.split():
        if w.upper() in chat_words:
            new_text.append(chat_words[w.upper()])
        else:
            new_text.append(w)
        
    return ' '.join(new_text)    

In [27]:
chat_conversion('IMHO he is the best')

'In My Honest/Humble Opinion he is the best'

In [28]:
chat_conversion('FYI delhi is the capital of India')

'For Your Information delhi is the capital of India'

<h2> 6. Spelling Correction </h2>

In [29]:
from textblob import TextBlob

In [30]:
incorrect_text = "Machine learnning is a branch of artifecial intelligence and computer sciance."

In [31]:
textBlb = TextBlob(incorrect_text)
textBlb.correct().string

'Machine learning is a branch of artificial intelligence and computer science.'

<h2> 7. Removing Stopwords </h2>

In [32]:
from nltk.corpus import stopwords

In [33]:
stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [34]:
def remove_stopwords(text):
    new_text = []

    for word in text.split():
        if word in stopwords.words('english'):
            new_text.append('')
        else:
            new_text.append(word)

    x = new_text[:]
    new_text.clear()
        
    return " ".join(x)    

In [35]:
remove_stopwords("This is a sample sentence, showing off the stop words filtration process.")

'This   sample sentence, showing   stop words filtration process.'

<h2> 8. Removing Emojis </h2>

In [37]:
import re

def remove_emoji(text):
    emoji_pattern = re.compile('[\U00010000-\U0010ffff]', flags=re.UNICODE)

    return emoji_pattern.sub(r'', text)

In [38]:
remove_emoji("I love NLP! 😍 It's amazing 🚀.")

"I love NLP!  It's amazing ."

In [39]:
remove_emoji("Wow! 😮 That is fast 🚀. 👍")

'Wow!  That is fast . '

In [40]:
import emoji

print(emoji.demojize('I ❤️ python'))

I :red_heart: python


In [41]:
print(emoji.demojize("I feel great! 😊 Let's celebrate! 🎉"))

I feel great! :smiling_face_with_smiling_eyes: Let's celebrate! :party_popper:
